# Pace Regression: XGBoost vs Random Forest
Predict next-event pace per lap using features from previous runs.

*Co-authored with CoCo*

In [1]:
# from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# session = get_active_session()
# print('Session ready')

In [5]:
# Build features: current metrics, lag values, rolling avg, and condition category

df = pd.read_csv("STAGE_RUNPROJECT.csv")
df['DATE'] = pd.to_datetime(df['DATE'])

# Sort for proper lag computation
df = df.sort_values(['CONDITION_CATEGORY', 'LAPS', 'DATE']).reset_index(drop=True)

# Create lag features per (CONDITION_CATEGORY, LAPS) group
group_cols = ['CONDITION_CATEGORY', 'LAPS']

df['LAG_PACE'] = df.groupby(group_cols)['AVG_PACE_MIN_KM'].shift(1)
df['LAG2_PACE'] = df.groupby(group_cols)['AVG_PACE_MIN_KM'].shift(2)
df['ROLLING_AVG_PACE'] = df.groupby(group_cols)['AVG_PACE_MIN_KM'].transform(
    lambda x: x.expanding().mean().shift(1))
df['LAG_HR'] = df.groupby(group_cols)['AVG_HR_BPM'].shift(1)
df['LAG_CADENCE'] = df.groupby(group_cols)['AVG_RUN_CADENCE_SPM'].shift(1)
df['LAG_STRIDE'] = df.groupby(group_cols)['AVG_STRIDE_LENGTH_M'].shift(1)
df['LAG_CUM_TIME'] = df.groupby(group_cols)['CUMULATIVE_TIME'].shift(1)
df['LAG_ASCENT'] = df.groupby(group_cols)['TOTAL_ASCENT_M'].shift(1)

# Date sequence number per group (for trend)
df['DATE_SEQ'] = df.groupby(group_cols).cumcount()

# Define all features: current metrics + lag + rolling + category
target = 'AVG_PACE_MIN_KM'
features = [
    # Current run metrics
    'LAPS', 'AVG_HR_BPM', 'AVG_RUN_CADENCE_SPM', 'AVG_STRIDE_LENGTH_M',
    'CUMULATIVE_TIME', 'TOTAL_ASCENT_M',
    # Condition category (weighted via sample_weight during training)
    'CONDITION_CATEGORY',
    # Lag features
    'LAG_PACE', 'LAG2_PACE', 'ROLLING_AVG_PACE',
    'LAG_HR', 'LAG_CADENCE', 'LAG_STRIDE', 'LAG_CUM_TIME', 'LAG_ASCENT',
    # Trend
    'DATE_SEQ'
]

# Show feature ratings: correlation with target
df_valid = df.dropna(subset=['LAG_PACE']).copy()
corr = df_valid[features + [target]].corr()[target].drop(target).abs().sort_values(ascending=False)
print('=== FEATURE RATINGS (|correlation| with target pace) ===')
for feat, score in corr.items():
    bar = '█' * int(score * 30)
    print(f'  {feat:<25s} {score:.3f}  {bar}')
print(f'\nTotal features: {len(features)}')

=== FEATURE RATINGS (|correlation| with target pace) ===
  AVG_RUN_CADENCE_SPM       0.961  ████████████████████████████
  AVG_HR_BPM                0.817  ████████████████████████
  AVG_STRIDE_LENGTH_M       0.656  ███████████████████
  CUMULATIVE_TIME           0.556  ████████████████
  ROLLING_AVG_PACE          0.532  ███████████████
  LAG_CUM_TIME              0.473  ██████████████
  LAPS                      0.453  █████████████
  LAG_CADENCE               0.422  ████████████
  LAG_PACE                  0.416  ████████████
  LAG_HR                    0.358  ██████████
  LAG_STRIDE                0.325  █████████
  LAG2_PACE                 0.283  ████████
  CONDITION_CATEGORY        0.274  ████████
  TOTAL_ASCENT_M            0.235  ███████
  LAG_ASCENT                0.220  ██████
  DATE_SEQ                  0.164  ████

Total features: 16


In [7]:
# Split: latest date per condition category is eval set
latest_dates = df.groupby('CONDITION_CATEGORY')['DATE'].max().reset_index()
latest_dates.columns = ['CONDITION_CATEGORY', 'LATEST_DATE']
print('Eval dates (latest event per category):')
print(latest_dates)

df_valid = df.dropna(subset=['LAG_PACE']).copy()
df_valid = df_valid.merge(latest_dates, on='CONDITION_CATEGORY')

train = df_valid[df_valid['DATE'] < df_valid['LATEST_DATE']].copy()
eval_set = df_valid[df_valid['DATE'] == df_valid['LATEST_DATE']].copy()

# Fill remaining NaN in LAG2 with LAG_PACE
train['LAG2_PACE'] = train['LAG2_PACE'].fillna(train['LAG_PACE'])
eval_set['LAG2_PACE'] = eval_set['LAG2_PACE'].fillna(eval_set['LAG_PACE'])

X_train = train[features]
y_train = train[target]
X_eval = eval_set[features]
y_eval = eval_set[target]

# Create sample weights: upweight CONDITION_CATEGORY=1 (events) since they are rarer
category_weights = {0: 1.0, 1: 2.0}
train_weights = train['CONDITION_CATEGORY'].map(category_weights).values

print(f'\nTrain size: {len(X_train)} rows')
print(f'Eval size: {len(X_eval)} rows')
print(f'Category weight map: {category_weights}')

Eval dates (latest event per category):
   CONDITION_CATEGORY LATEST_DATE
0                   0  2026-06-18
1                   1  2026-06-21

Train size: 80 rows
Eval size: 20 rows
Category weight map: {0: 1.0, 1: 2.0}


In [ ]:
print(X_eval)

In [8]:
# Train XGBoost with sample weights for CONDITION_CATEGORY
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    min_child_weight=3,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42
)
xgb_model.fit(X_train, y_train, sample_weight=train_weights)
xgb_preds = xgb_model.predict(X_eval)

# Train Random Forest with sample weights
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=3,
    random_state=42
)
rf_model.fit(X_train, y_train, sample_weight=train_weights)
rf_preds = rf_model.predict(X_eval)

# Train HistGradientBoosting (handles NaN natively, no need to fill)
hgb_model = HistGradientBoostingRegressor(
    max_iter=100,
    max_depth=3,
    learning_rate=0.1,
    min_samples_leaf=3,
    l2_regularization=1.0,
    random_state=42
)
hgb_model.fit(X_train, y_train, sample_weight=train_weights)
hgb_preds = hgb_model.predict(X_eval)

# Time-series forecast: Holt-Winters per (CONDITION_CATEGORY, LAPS) group
# Predict the next pace value using exponential smoothing on historical pace sequence
ts_preds = []
for _, row in eval_set.iterrows():
    cat = row['CONDITION_CATEGORY']
    lap = row['LAPS']
    # Get historical pace for this group in training data
    history = train[(train['CONDITION_CATEGORY'] == cat) & (train['LAPS'] == lap)]['AVG_PACE_MIN_KM'].values
    if len(history) >= 3:
        model_ts = ExponentialSmoothing(history, trend='add', seasonal=None).fit(optimized=True)
        ts_preds.append(model_ts.forecast(1)[0])
    elif len(history) >= 1:
        ts_preds.append(history.mean())
    else:
        ts_preds.append(np.nan)

ts_preds = np.array(ts_preds)
print('All models trained: XGBoost, Random Forest, HistGradientBoosting, Time-Series (Holt-Winters)')

All models trained: XGBoost, Random Forest, HistGradientBoosting, Time-Series (Holt-Winters)


In [9]:
# Evaluate all models on latest event (eval set)
def evaluate(y_true, y_pred, name):
    mask = ~np.isnan(y_pred)
    mae = mean_absolute_error(y_true[mask], y_pred[mask])
    rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
    r2 = r2_score(y_true[mask], y_pred[mask])
    print(f'{name}:')
    print(f'  MAE:  {mae:.3f} min/km')
    print(f'  RMSE: {rmse:.3f} min/km')
    print(f'  R2:   {r2:.3f}')
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

print('=== MODEL COMPARISON (latest event as eval set) ===')
print()
xgb_metrics = evaluate(y_eval.values, xgb_preds, 'XGBoost')
print()
rf_metrics = evaluate(y_eval.values, rf_preds, 'Random Forest')
print()
hgb_metrics = evaluate(y_eval.values, hgb_preds, 'HistGradientBoosting')
print()
ts_metrics = evaluate(y_eval.values, ts_preds, 'Time-Series (Holt-Winters)')
print()

# Summary table
all_metrics = [xgb_metrics, rf_metrics, hgb_metrics, ts_metrics]
results_df = pd.DataFrame(all_metrics).sort_values('MAE')
print('=== RANKING (by MAE) ===')
print(results_df.to_string(index=False))
print()

best = results_df.iloc[0]['Model']
print(f'>>> BEST MODEL: {best} <<<')

model_map = {'XGBoost': xgb_model, 'Random Forest': rf_model, 'HistGradientBoosting': hgb_model}
best_model = model_map.get(best, xgb_model)

=== MODEL COMPARISON (latest event as eval set) ===

XGBoost:
  MAE:  0.408 min/km
  RMSE: 0.600 min/km
  R2:   0.897

Random Forest:
  MAE:  0.581 min/km
  RMSE: 0.848 min/km
  R2:   0.795

HistGradientBoosting:
  MAE:  0.341 min/km
  RMSE: 0.555 min/km
  R2:   0.912

Time-Series (Holt-Winters):
  MAE:  1.910 min/km
  RMSE: 2.466 min/km
  R2:   -0.492

=== RANKING (by MAE) ===
                     Model      MAE     RMSE        R2
      HistGradientBoosting 0.341335 0.554585  0.912368
                   XGBoost 0.407864 0.600172  0.897369
             Random Forest 0.581493 0.847816  0.795201
Time-Series (Holt-Winters) 1.909964 2.466366 -0.491811

>>> BEST MODEL: HistGradientBoosting <<<


In [10]:
# Predict next-event pace for laps 1-10 using the best model
# Use the latest observation per lap as input features

def predict_next_event(model, df_source, condition_cat, features):
    latest = df_source[
        df_source['CONDITION_CATEGORY'] == condition_cat
    ].sort_values(['LAPS', 'DATE']).groupby('LAPS').last().reset_index()

    pred_input = pd.DataFrame({
        'LAPS': latest['LAPS'],
        # Current metrics: use latest known values as proxy for next event
        'AVG_HR_BPM': latest['AVG_HR_BPM'],
        'AVG_RUN_CADENCE_SPM': latest['AVG_RUN_CADENCE_SPM'],
        'AVG_STRIDE_LENGTH_M': latest['AVG_STRIDE_LENGTH_M'],
        'CUMULATIVE_TIME': latest['CUMULATIVE_TIME'],
        'TOTAL_ASCENT_M': latest['TOTAL_ASCENT_M'],
        'CONDITION_CATEGORY': condition_cat,
        # Lag features: latest actual values become the lag for next prediction
        'LAG_PACE': latest['AVG_PACE_MIN_KM'],
        'LAG2_PACE': latest['LAG_PACE'].fillna(latest['AVG_PACE_MIN_KM']),
        'ROLLING_AVG_PACE': latest['ROLLING_AVG_PACE'],
        'LAG_HR': latest['AVG_HR_BPM'],
        'LAG_CADENCE': latest['AVG_RUN_CADENCE_SPM'],
        'LAG_STRIDE': latest['AVG_STRIDE_LENGTH_M'],
        'LAG_CUM_TIME': latest['CUMULATIVE_TIME'],
        'LAG_ASCENT': latest['TOTAL_ASCENT_M'],
        'DATE_SEQ': latest['DATE_SEQ'] + 1
    })
    pred_input['ROLLING_AVG_PACE'] = pred_input['ROLLING_AVG_PACE'].fillna(pred_input['LAG_PACE'])

    preds = model.predict(pred_input[features])
    return pd.DataFrame({'LAPS': pred_input['LAPS'].values, 'PRED_PACE': np.round(preds, 2)})

result_cat0 = predict_next_event(best_model, df_valid, 0, features)
result_cat1 = predict_next_event(best_model, df_valid, 1, features)

print(f'=== PREDICTION: CONDITION_CATEGORY = 0 (Regular Training) ===')
print(f'Model: {best}')
print(result_cat0.to_string(index=False))

print(f'\n=== PREDICTION: CONDITION_CATEGORY = 1 (Event Performance) ===')
print(f'Model: {best}')
print(result_cat1.to_string(index=False))

=== PREDICTION: CONDITION_CATEGORY = 0 (Regular Training) ===
Model: HistGradientBoosting
 LAPS  PRED_PACE
  1.0      13.97
  2.0       7.34
  3.0       7.54
  4.0       7.51
  5.0       7.75
  6.0       7.40
  7.0       7.76
  8.0       9.16
  9.0      12.81
 10.0      11.60

=== PREDICTION: CONDITION_CATEGORY = 1 (Event Performance) ===
Model: HistGradientBoosting
 LAPS  PRED_PACE
  1.0       6.83
  2.0       6.62
  3.0       6.85
  4.0       7.64
  5.0       6.63
  6.0       6.73
  7.0       7.60
  8.0       8.93
  9.0       9.05
 10.0       7.92


In [ ]:
# Generate predictions from all ML models for both condition categories
# Write combined results to a temp table for SQL charting
from sklearn.ensemble import HistGradientBoostingRegressor

models = {'XGBoost': xgb_model, 'RandomForest': rf_model, 'HistGradientBoosting': hgb_model}
category_labels = {0: 'Regular Training', 1: 'Event Performance'}

all_preds = []
for cat in [0, 1]:
    for model_name, model in models.items():
        preds_df = predict_next_event(model, df_valid, cat, features)
        preds_df['CONDITION_CATEGORY'] = cat
        preds_df['CATEGORY_LABEL'] = category_labels[cat]
        preds_df['MODEL'] = model_name
        preds_df['LINE_LABEL'] = f'{category_labels[cat]} - {model_name}'
        all_preds.append(preds_df)

chart_df = pd.concat(all_preds, ignore_index=True)
chart_df = chart_df.rename(columns={'PRED_PACE': 'PACE'})

# Write to temp table for SQL charting
# session.write_pandas(chart_df, table_name='PRED_CHART_DATA', database='RUN_PROJECT', schema='PUBLIC', auto_create_table=True, overwrite=True)
chart_df.to_csv('PRED_CHART_DATA.csv', index=False)
print(f'Wrote {len(chart_df)} rows to RUN_PROJECT.PUBLIC.PRED_CHART_DATA')
print(chart_df.head(10).to_string(index=False))

NameError: name 'session' is not defined

In [12]:
# Retrain HistGradientBoosting on ALL data (including eval rows) for production prediction
# This uses the full dataset - no hold-out

df_full = df.dropna(subset=['LAG_PACE']).copy()
df_full['LAG2_PACE'] = df_full['LAG2_PACE'].fillna(df_full['LAG_PACE'])

X_full = df_full[features]
y_full = df_full[target]

hgb_final = HistGradientBoostingRegressor(
    max_iter=150,
    max_depth=3,
    learning_rate=0.08,
    min_samples_leaf=3,
    l2_regularization=1.0,
    random_state=42
)

# Weight events higher
full_weights = df_full['CONDITION_CATEGORY'].map({0: 1.0, 1: 2.0}).values
hgb_final.fit(X_full, y_full, sample_weight=full_weights)

# Predict next event using ALL available history
result_cat0_final = predict_next_event(hgb_final, df_full, 0, features)
result_cat1_final = predict_next_event(hgb_final, df_full, 1, features)

print('=== FINAL PREDICTION (HistGradientBoosting trained on ALL data) ===')
print(f'Training rows: {len(X_full)} (eval rows added back)')
print()
print('CONDITION_CATEGORY = 0 (Regular Training):')
print(result_cat0_final.to_string(index=False))
print()
print('CONDITION_CATEGORY = 1 (Event Performance):')
print(result_cat1_final.to_string(index=False))

=== FINAL PREDICTION (HistGradientBoosting trained on ALL data) ===
Training rows: 100 (eval rows added back)

CONDITION_CATEGORY = 0 (Regular Training):
 LAPS  PRED_PACE
  1.0      13.17
  2.0       7.34
  3.0       7.55
  4.0       7.48
  5.0       7.79
  6.0       7.38
  7.0       7.79
  8.0       8.99
  9.0      11.40
 10.0      11.27

CONDITION_CATEGORY = 1 (Event Performance):
 LAPS  PRED_PACE
  1.0       5.93
  2.0       5.83
  3.0       6.33
  4.0       7.16
  5.0       6.31
  6.0       6.75
  7.0       7.70
  8.0       8.49
  9.0       8.98
 10.0       7.99
